> **Source notebook** (nemotronTASS Lab 23, outputs stripped) that identity-liaison was built from. Kept for reference; the installable tool is in `../identity_liaison/`.


# Lab 23 — Download measured reply latency as CSV
**One improvement:** Export successful TASS browser request durations as CSV; no simulated measurements are exported.

**Prerequisites:** Windows/WSL, Python 3, Jupyter, local llama.cpp Nemotron 3 Nano 4B at `127.0.0.1:8081`.

1. In an isolated checkout run `MODEL_PATH="$HOME/nemotronTass/models/NVIDIA-Nemotron3-Nano-4B-Q4_K_M.gguf" ./run.sh lab23`.
2. Run **Start TASS** in Jupyter; open **http://127.0.0.1:8784** in Windows browser.
3. Send `Hello TASS`, wait for a real visible reply, click **Download latency CSV**, and open `tass-reply-latency.csv`.
4. Verify header `reply_number,request_seconds` and one numeric nonnegative duration row. Click **Stop companion** or use fallback Stop cell.

**One repeatable acceptance test:** Actual reply visible → CSV has measured row → Stop prevents new requests. All required for PASS.

**Troubleshooting:** `curl http://127.0.0.1:8081/health`; stop previous companion if 8784 busy; check browser downloads and WSL localhost forwarding. Failed model requests create no CSV row.

Loopback only; browser-memory conversation; no paid API, training, saved secrets or autonomous edits.


## Start TASS

In [ ]:
import json, threading, urllib.request, urllib.error
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
MODEL_URL='http://127.0.0.1:8081/v1/chat/completions'
HOST,PORT='127.0.0.1',8784
HTML="<!doctype html><meta charset=\"utf-8\"><title>TASS Lab 22</title><style>body{background:#14202c;color:white;font:16px system-ui;max-width:700px;margin:3em auto}article{padding:12px;border:1px solid #789;margin:8px 0}button,input{padding:10px;font:inherit}</style><h1>TASS Lab 22</h1><div id=\"chat\"></div><p id=\"status\">Ready</p><form id=\"form\"><input id=\"message\" required maxlength=\"4000\" placeholder=\"Talk to TASS\"><button id=\"send\">Send</button></form><button id=\"stop\">Stop companion</button><button id=\"export\" type=\"button\">Download latency CSV</button><script>\nconst latencies=[],turns=[],chat=document.getElementById('chat'),status=document.getElementById('status'),send=document.getElementById('send'),input=document.getElementById('message');\ndocument.getElementById('form').onsubmit=async e=>{e.preventDefault();const message=input.value.trim();if(!message)return;send.disabled=true;const requestStarted=performance.now();status.textContent='Waiting for Nemotron';try{const r=await fetch('/api/chat',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({messages:[...turns,{role:'user',content:message}]})});const d=await r.json();if(!r.ok||!d.reply)throw Error(d.error||'Empty reply');turns.push({role:'user',content:message},{role:'assistant',content:d.reply});chat.replaceChildren();for(const t of turns){const a=document.createElement('article');a.textContent=(t.role==='user'?'You: ':'TASS: ')+t.content;chat.append(a)}input.value='';const seconds=(performance.now()-requestStarted)/1000;latencies.push({turn:latencies.length+1,seconds:seconds.toFixed(3)});status.textContent='Reply received in '+seconds.toFixed(1)+' s'}catch(err){status.textContent='FAIL: '+err.message}finally{send.disabled=false}};\ndocument.getElementById('export').onclick=()=>{if(!latencies.length){status.textContent='No successful replies to export';return;}const csv='reply_number,request_seconds\\n'+latencies.map(x=>x.turn+','+x.seconds).join('\\n')+'\\n';const url=URL.createObjectURL(new Blob([csv],{type:'text/csv;charset=utf-8'}));const a=document.createElement('a');a.href=url;a.download='tass-reply-latency.csv';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);status.textContent='Exported '+latencies.length+' measured replies';};\\ndocument.getElementById('stop').onclick=async()=>{if(!confirm('Stop companion?'))return;try{const r=await fetch('/api/stop',{method:'POST'});if(!r.ok)throw Error('Stop rejected');status.textContent='Companion stopped';send.disabled=true}catch(err){status.textContent='Stop failed: '+err.message}};\n</script>"
server=None
class Handler(BaseHTTPRequestHandler):
    def log_message(self,*args): pass
    def respond(self,code,value):
        b=json.dumps(value).encode();self.send_response(code);self.send_header('Content-Type','application/json');self.send_header('Content-Length',str(len(b)));self.end_headers();self.wfile.write(b)
    def do_GET(self):
        if self.path!='/': return self.respond(404,{'error':'Not found'})
        b=HTML.encode();self.send_response(200);self.send_header('Content-Type','text/html; charset=utf-8');self.send_header('Content-Length',str(len(b)));self.end_headers();self.wfile.write(b)
    def do_POST(self):
        if self.path=='/api/stop':
            self.respond(200,{'stopped':True});threading.Thread(target=lambda:(self.server.shutdown(),self.server.server_close()),daemon=True).start();return
        if self.path!='/api/chat': return self.respond(404,{'error':'Not found'})
        try:
            n=int(self.headers.get('Content-Length','0'))
            if not 0<n<=65536: raise ValueError('Invalid size')
            messages=json.loads(self.rfile.read(n))['messages']
            if not isinstance(messages,list) or not 1<=len(messages)<=40: raise ValueError('Expected 1–40 messages')
            if any(not isinstance(m,dict) or m.get('role') not in ('user','assistant') or not isinstance(m.get('content'),str) or not 0<len(m['content'])<=4000 for m in messages): raise ValueError('Invalid message')
            req=urllib.request.Request(MODEL_URL,data=json.dumps({'model':'local','messages':messages,'stream':False,'max_tokens':300}).encode(),headers={'Content-Type':'application/json'})
            with urllib.request.urlopen(req,timeout=90) as response: data=json.load(response)
            answer=data['choices'][0]['message']['content']
            if not isinstance(answer,str) or not answer.strip(): raise ValueError('Empty reply')
            self.respond(200,{'reply':answer})
        except (ValueError,KeyError,IndexError,TypeError,urllib.error.URLError,TimeoutError) as exc: self.respond(502,{'error':str(exc)[:200]})
        except Exception as exc: self.respond(502,{'error':type(exc).__name__})
if server is not None: server.shutdown();server.server_close()
server=ThreadingHTTPServer((HOST,PORT),Handler)
threading.Thread(target=server.serve_forever,daemon=True).start()
print(f'Open http://{HOST}:{PORT} — use Stop companion or Stop cell')


## Offline integration test (fake model only)

In [ ]:
# Fake-model integration test; never counted as real Nemotron.
import json, threading, urllib.request, urllib.error
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
class Fake(BaseHTTPRequestHandler):
    def log_message(self,*args): pass
    def do_POST(self):
        n=int(self.headers['Content-Length']);json.loads(self.rfile.read(n))
        b=json.dumps({'choices':[{'message':{'content':'fixture reply'}}]}).encode()
        self.send_response(200);self.send_header('Content-Length',str(len(b)));self.end_headers();self.wfile.write(b)
fake=ThreadingHTTPServer(('127.0.0.1',0),Fake)
threading.Thread(target=fake.serve_forever,daemon=True).start()
original=MODEL_URL
try:
    MODEL_URL=f'http://127.0.0.1:{fake.server_port}/v1/chat/completions'
    with urllib.request.urlopen(f'http://{HOST}:{PORT}/',timeout=5) as r: assert b'Download latency CSV' in r.read()
    req=urllib.request.Request(f'http://{HOST}:{PORT}/api/chat',data=json.dumps({'messages':[{'role':'user','content':'Hello TASS'}]}).encode(),headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req,timeout=5) as r: assert json.load(r)['reply']=='fixture reply'
    MODEL_URL='http://127.0.0.1:1/v1/chat/completions'
    try: urllib.request.urlopen(req,timeout=5);raise AssertionError('Expected 502')
    except urllib.error.HTTPError as e: assert e.code==502
    print('PASS: offline HTML, fake reply, disconnected-model error')
finally:
    MODEL_URL=original;fake.shutdown();fake.server_close()


## Stop companion (fallback)

In [ ]:
if server is not None:
    server.shutdown();server.server_close();server=None
    print('Companion stopped')

## Owner result record

- Overall: **NOT RUN**
- Date/time:
- Real Nemotron reply: PASS / FAIL / NOT RUN
- CSV download and nonnegative measured row: PASS / FAIL / NOT RUN
- Stop companion: PASS / FAIL / NOT RUN
- Actual CSV/error output (verbatim):

Offline fixture results are not evidence of a live Windows browser or model.


In [ ]:

from labkit import Nemotron
import ipywidgets as widgets
from IPython.display import display, clear_output

nemo = Nemotron()

question = widgets.Textarea(
    value="I have 3 tasks that each take 2 hours, but only 5 hours available. What should I do?",
    description="Question:",
    layout=widgets.Layout(width="95%", height="90px")
)

think_btn = widgets.Button(
    description="Thinking ON",
    button_style="success"
)
fast_btn = widgets.Button(
    description="Thinking OFF",
    button_style="info"
)

output = widgets.Output()

def run_model(thinking):
    with output:
        clear_output(wait=True)
        print("THINKING ON" if thinking else "THINKING OFF")
        print("-" * 50)
        try:
            nemo.ask(question.value, thinking=thinking)
        except Exception as error:
            print("ERROR:", error)

think_btn.on_click(lambda _: run_model(True))
fast_btn.on_click(lambda _: run_model(False))

display(question)
display(widgets.HBox([think_btn, fast_btn]))
display(output)

In [ ]:
import subprocess
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

REPO = "Arameair/nemotronTASS"

connect_btn = widgets.Button(
    description="Connect to GitHub",
    button_style="success",
    icon="github"
)

output = widgets.Output()

def github_command(*args):
    result = subprocess.run(
        ["gh", *args],
        capture_output=True,
        text=True,
        timeout=20
    )
    if result.returncode != 0:
        raise RuntimeError("GitHub command failed. Check authentication.")
    return result.stdout

def connect_github(_):
    with output:
        clear_output(wait=True)

        try:
            github_command("auth", "status")

            repo = json.loads(github_command(
                "api", f"repos/{REPO}"
            ))

            issues = json.loads(github_command(
                "api", "-X", "GET",
                f"repos/{REPO}/issues",
                "-f", "state=open",
                "-f", "per_page=10"
            ))

            print("CONNECTED TO GITHUB")
            print("Repository:", repo["full_name"])
            print("Private:", repo["private"])
            print("\nOpen issues:")

            for issue in issues:
                if "pull_request" not in issue:
                    print(f"#{issue['number']} — {issue['title']}")

            print("\nRead-only connection verified.")

        except FileNotFoundError:
            print("GitHub CLI (gh) is not installed.")
        except Exception as error:
            print("Connection failed:", error)
            print("Run 'gh auth login' in your WSL terminal if needed.")

connect_btn.on_click(connect_github)

display(connect_btn, output)

In [ ]:
import subprocess
import shutil
import json

REPO = "Arameair/nemotronTASS"

def check(name, command):
    print(f"\n=== {name} ===")
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=20
        )

        print("Exit code:", result.returncode)

        if result.returncode != 0:
            print("Error:", result.stderr.strip()[:300])
            return None

        output = result.stdout.strip()

        if not output:
            print("FAIL: Command returned empty output.")
            return None

        print("PASS:", output[:300])
        return output

    except Exception as e:
        print("FAIL:", type(e).__name__, str(e))
        return None

print("GitHub CLI:", shutil.which("gh") or "NOT FOUND")

check(
    "GitHub authentication",
    ["gh", "api", "user", "--jq", ".login"]
)

check(
    "Repository access",
    ["gh", "api", f"repos/{REPO}", "--jq", ".full_name"]
)

check(
    "Read GitHub issues",
    [
        "gh", "api", "--method", "GET",
        f"repos/{REPO}/issues",
        "-f", "state=open",
        "-f", "per_page=5",
        "--jq", ".[].title"
    ]
)


In [ ]:
import subprocess
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

REPO = "Arameair/nemotronTASS"
BRANCH = "experiment/tass-foundation-notebook-20261103"
FILE = "TASS.md"

read_button = widgets.Button(
    description="Read TASS.md from GitHub",
    button_style="success",
    icon="book"
)

output = widgets.Output()

def read_tass(_):
    with output:
        clear_output(wait=True)

        try:
            result = subprocess.run(
                [
                    "gh", "api",
                    f"repos/{REPO}/contents/{FILE}",
                    "-f", f"ref={BRANCH}",
                    "--method", "GET",
                    "--jq", ".content"
                ],
                capture_output=True,
                text=True,
                timeout=20,
                check=True
            )

            import base64

            encoded = "".join(result.stdout.split())
            tass_text = base64.b64decode(
                encoded,
                validate=True
            ).decode("utf-8")

            if not tass_text.strip():
                raise ValueError("TASS.md is empty")

            globals()["TASS_CONTEXT"] = tass_text

            print("PASS: TASS.md loaded from GitHub")
            print("Characters:", len(tass_text))
            print("Saved to: TASS_CONTEXT")
            print("-" * 50)

            display(Markdown(tass_text))

        except Exception as error:
            print("FAIL:", str(error))

read_button.on_click(read_tass)

display(read_button, output)


In [ ]:
import subprocess
import json
import base64
from IPython.display import display, Markdown

REPO = "Arameair/nemotronTASS"
BRANCH = "experiment/tass-foundation-notebook-20261103"

cmd = [
    "gh", "api",
    "--method", "GET",
    f"repos/{REPO}/contents/TASS.md",
    "-f", f"ref={BRANCH}",
    "-H", "Accept: application/vnd.github+json"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    timeout=30
)

print("GitHub exit code:", result.returncode)

if result.returncode != 0:
    print("FAIL:", result.stderr[:500])

elif not result.stdout.strip():
    print("FAIL: Empty response")

else:
    try:
        data = json.loads(result.stdout)

        TASS_CONTEXT = base64.b64decode(
            data["content"].replace("\n", "")
        ).decode("utf-8")

        print("PASS: TASS.md loaded")
        print("Characters:", len(TASS_CONTEXT))
        display(Markdown(TASS_CONTEXT))

    except (json.JSONDecodeError, KeyError) as error:
        print("Response format:", type(error).__name__)
        print("First 200 characters:")
        print(repr(result.stdout[:200]))

In [ ]:
import os

# Disable ANSI colors for GitHub CLI subprocesses
os.environ["CLICOLOR"] = "0"
os.environ["NO_COLOR"] = "1"
os.environ["GH_FORCE_TTY"] = "0"

print("GitHub CLI color output disabled.")

In [ ]:

import json
import urllib.request
import urllib.error
import ipywidgets as widgets
from IPython.display import display, clear_output

MODEL = "http://127.0.0.1:8081"

# Reserve capacity for the answer and uncertainty.
RESPONSE_TOKENS = 768
SAFETY_MARGIN = 256

def model_api(path, payload=None):
    data = None if payload is None else json.dumps(
        payload
    ).encode("utf-8")

    request = urllib.request.Request(
        MODEL + path,
        data=data,
        headers={"Content-Type": "application/json"}
    )

    with urllib.request.urlopen(request, timeout=20) as r:
        return json.load(r)

def context_guard(messages, show=True):
    """
    Check context capacity before sending a model request.
    Returns True only when the request passes.
    """

    try:
        # 1. Read actual running context capacity.
        props = model_api("/props")

        capacity = props.get(
            "default_generation_settings", {}
        ).get("n_ctx")

        if not isinstance(capacity, int) or capacity <= 0:
            raise ValueError(
                "Server did not report a usable context limit"
            )

        # 2. Apply the model's chat template.
        formatted = model_api(
            "/apply-template",
            {"messages": messages}
        )

        prompt = formatted.get("prompt")

        if not isinstance(prompt, str):
            raise ValueError("Chat template unavailable")

        # 3. Count actual tokenizer tokens.
        result = model_api(
            "/tokenize",
            {"content": prompt, "add_special": True}
        )

        tokens = result.get("tokens")

        if not isinstance(tokens, list):
            raise ValueError("Tokenizer unavailable")

        used = len(tokens)

        # 4. Reserve response space and safety margin.
        available = (
            capacity - RESPONSE_TOKENS - SAFETY_MARGIN
        )

        passed = available >= 0 and used <= available

        if show:
            print("=== TASS CONTEXT GUARD ===")
            print("Endpoint:", MODEL)
            print("Context capacity:", capacity)
            print("Prompt tokens:", used)
            print("Response reserve:", RESPONSE_TOKENS)
            print("Safety margin:", SAFETY_MARGIN)
            print("Input budget:", max(0, available))
            print("Input remaining:", max(0, available - used))

            if passed:
                print("\nPASS: Request fits context budget.")
            else:
                print("\nBLOCKED: Context budget exceeded.")
                print("Reduce context before sending.")

        return passed

    except Exception as e:
        if show:
            print("BLOCKED: Cannot verify context budget.")
            print("Reason:", str(e))
        return False

# Notebook controls
check_button = widgets.Button(
    description="Check TASS Context",
    button_style="info",
    icon="shield"
)

test_button = widgets.Button(
    description="Test Oversized Context",
    button_style="warning"
)

output = widgets.Output()

def check_tass(_):
    with output:
        clear_output(wait=True)

        if "TASS_CONTEXT" not in globals():
            print("BLOCKED: TASS_CONTEXT is not loaded.")
            print("Run the GitHub TASS.md reading cell first.")
            return

        messages = [
            {
                "role": "system",
                "content": "You are TASS's local development model."
            },
            {
                "role": "user",
                "content": (
                    "Read this TASS definition:\n\n"
                    + TASS_CONTEXT
                    + "\n\nExplain your role."
                )
            }
        ]

        context_guard(messages)

def test_oversized(_):
    with output:
        clear_output(wait=True)

        messages = [{
            "role": "user",
            "content": "Test data. " * 20000
        }]

        result = context_guard(messages)

        if result:
            print("TEST: FAIL — unexpected acceptance.")
        else:
            print("TEST: Oversized request not approved.")

check_button.on_click(check_tass)
test_button.on_click(test_oversized)

display(widgets.HBox([check_button, test_button]))
display(output)

In [ ]:

import json
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# TASS Core Team — example records only.
# No actual workers, tasks, or evidence are verified.

TEAM_ID = "tass-core"
IDENTITY = "TASS"

ROLE_DEFINITION = {
    "role": "Team Liaison",
    "mission": (
        "Maintain trustworthy communication between authorized "
        "team participants. Track acknowledgments, evidence, "
        "blockers, and permitted handoffs without assuming "
        "project leadership."
    ),
    "responsibilities": [
        "Read authorized team communication records",
        "Track delivery and acknowledgment separately",
        "Identify missing responses and blockers",
        "Request evidence for completion claims",
        "Escalate assignment conflicts to Project Lead",
        "Avoid unnecessary or duplicate nudges"
    ],
    "prohibited": [
        "Create new project scope",
        "Approve permissions",
        "Assign work without authorization",
        "Treat DONE as verified evidence",
        "Control another identity's team"
    ],
    "protocol": [
        "DISPATCH: approved task and requirements",
        "WORKING: worker acknowledgment",
        "DONE: result and evidence claim",
        "BLOCKED: reason and required action"
    ],
    "pulse_minutes": 20,
    "max_nudges_per_pulse": 5
}

# Example worker communication cards.
CARDS = {
    "liaison": {
        "role": "Team Liaison",
        "task_id": None,
        "activity": "NOT RUN",
        "evidence": "UNVERIFIED",
        "acknowledgment": "NONE",
        "last_verified": None,
        "last_nudge_id": None,
        "delivery_receipt": None,
        "next_action": "Read team cards"
    },
    "lead": {
        "role": "Project Lead",
        "task_id": "EXAMPLE-001",
        "activity": "WORKING (SIMULATED)",
        "evidence": "UNVERIFIED",
        "acknowledgment": "WORKING (SIMULATED)",
        "last_verified": None,
        "last_nudge_id": None,
        "delivery_receipt": None,
        "next_action": "Review assignment evidence"
    },
    "engineer": {
        "role": "Engineer",
        "task_id": "EXAMPLE-002",
        "activity": "WORKING (SIMULATED)",
        "evidence": "UNVERIFIED",
        "acknowledgment": "WORKING (SIMULATED)",
        "last_verified": None,
        "last_nudge_id": None,
        "delivery_receipt": None,
        "next_action": "Await test evidence"
    },
    "tester": {
        "role": "Test Architect",
        "task_id": "EXAMPLE-003",
        "activity": "BLOCKED (SIMULATED)",
        "evidence": "BLOCKED",
        "acknowledgment": "BLOCKED (SIMULATED)",
        "last_verified": None,
        "last_nudge_id": None,
        "delivery_receipt": None,
        "next_action": "Report blocker to Project Lead"
    },
    "ux": {
        "role": "UX Architect",
        "task_id": None,
        "activity": "UNKNOWN",
        "evidence": "UNVERIFIED",
        "acknowledgment": "NONE",
        "last_verified": None,
        "last_nudge_id": None,
        "delivery_receipt": None,
        "next_action": "No action without assignment"
    }
}

role_options = [
    ("Team Liaison", "liaison"),
    ("Project Lead", "lead"),
    ("Engineer", "engineer"),
    ("Test Architect", "tester"),
    ("UX Architect", "ux")
]

selector = widgets.Dropdown(
    options=role_options,
    description="Worker:",
    layout=widgets.Layout(width="320px")
)

read_button = widgets.Button(
    description="Read Card",
    button_style="info",
    icon="book"
)

definition_button = widgets.Button(
    description="Read Liaison Rules",
    icon="shield"
)

output = widgets.Output()

def read_card(_):
    with output:
        clear_output(wait=True)

        card = CARDS[selector.value]

        print("=== TASS CORE TEAM ===")
        print("MODE: EXAMPLE DATA — NOT LIVE")
        print("Identity:", IDENTITY)
        print("Role:", card["role"])
        print("-" * 45)

        for key, value in card.items():
            if key != "role":
                print(f"{key}: {value}")

        globals()["ACTIVE_TEAM_CARD"] = card.copy()

        print("\nPASS: Example card loaded")
        print("Saved as ACTIVE_TEAM_CARD")

def read_rules(_):
    with output:
        clear_output(wait=True)

        display(Markdown(
            "# Team Liaison — Role Definition\n\n"
            f"**Mission:** {ROLE_DEFINITION['mission']}\n\n"
            "## Responsibilities\n" +
            "".join(
                f"- {x}\n"
                for x in ROLE_DEFINITION["responsibilities"]
            ) +
            "\n## Prohibited Actions\n" +
            "".join(
                f"- {x}\n"
                for x in ROLE_DEFINITION["prohibited"]
            ) +
            "\n## Communication Protocol\n" +
            "".join(
                f"- {x}\n"
                for x in ROLE_DEFINITION["protocol"]
            )
        ))

        print("Pulse:", ROLE_DEFINITION["pulse_minutes"],
              "minutes")
        print("Maximum nudges per pulse:",
              ROLE_DEFINITION["max_nudges_per_pulse"])
        print("STATUS: DESIGN RULES — NOT ENFORCED")

        globals()["ACTIVE_ROLE_DEFINITION"] = (
            ROLE_DEFINITION.copy()
        )

read_button.on_click(read_card)
definition_button.on_click(read_rules)

display(widgets.VBox([
    widgets.HTML(
        "<h3>TASS — Core Team Card Reader</h3>"
        "<p>Example data only. No live workers connected.</p>"
    ),
    selector,
    widgets.HBox([read_button, definition_button]),
    output
]))

In [ ]:

import json
import urllib.request
import ipywidgets as widgets
from IPython.display import display, clear_output

MODEL_URL = "http://127.0.0.1:8081"

opening_message = (
    "Hello NemoTASS Liaison. I am the ChatGPT-side "
    "Team Liaison for this communication experiment. "
    "Please explain your responsibilities, identify "
    "what evidence you need before claiming a worker "
    "is active, and ask me one question about "
    "coordinating our teams. Do not invent worker status."
)

history = []
nemo_reply = None

status = widgets.HTML(
    "<b>STATUS:</b> READY — NOT RUN"
)

conversation = widgets.HTML(
    value=(
        "<div style='padding:15px;background:#172333;"
        "color:white;border-radius:10px'>"
        "<h3>Team Liaison Conversation</h3>"
        "<p>Waiting for first message...</p></div>"
    )
)

ask_btn = widgets.Button(
    description="Start Liaison Conversation",
    button_style="success",
    icon="play"
)

reply_box = widgets.Textarea(
    placeholder="Paste ChatGPT Liaison's reply here...",
    layout=widgets.Layout(width="100%", height="110px"),
    disabled=True
)

continue_btn = widgets.Button(
    description="Send Reply to NemoTASS",
    button_style="info",
    disabled=True
)

output = widgets.Output()

def request_nemotron(messages):
    # Fail closed if the Context Guard is unavailable.
    guard = globals().get("context_guard")
    if not callable(guard):
        raise RuntimeError(
            "Run the Context Guard cell first."
        )

    if not guard(messages, show=True):
        raise RuntimeError("Context Guard blocked request")

    payload = json.dumps({
        "model": "local",
        "messages": messages,
        "max_tokens": 768,
        "stream": False
    }).encode()

    req = urllib.request.Request(
        MODEL_URL + "/v1/chat/completions",
        data=payload,
        headers={"Content-Type": "application/json"}
    )

    with urllib.request.urlopen(req, timeout=180) as r:
        data = json.load(r)

    text = data["choices"][0]["message"]["content"]

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Nemotron returned no visible reply")

    return text.strip()

def render():
    from html import escape

    html = [
        "<div style='background:#111b29;color:#eef2f7;"
        "padding:16px;border-radius:12px'>"
        "<h3>Live Liaison Exchange</h3>"
    ]

    for sender, text in history:
        color = "#1d4ed8" if sender == "ChatGPT" else "#155e50"
        html.append(
            f"<div style='background:{color};"
            "padding:12px;margin:10px 0;"
            "border-radius:10px'>"
            f"<b>{escape(sender)}</b><hr>"
            f"<div style='white-space:pre-wrap'>"
            f"{escape(text)}</div></div>"
        )

    html.append("</div>")
    conversation.value = "".join(html)

def start(_):
    global nemo_reply, history

    with output:
        clear_output(wait=True)

        try:
            if "ROLE_DEFINITION" not in globals():
                raise RuntimeError(
                    "Run the Team Card Reader cell first."
                )

            status.value = "<b>STATUS:</b> Contacting Nemotron"
            ask_btn.disabled = True
            history = [("ChatGPT", opening_message)]
            render()

            rules = json.dumps(ROLE_DEFINITION)

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are simulating the TASS Team Liaison "
                        "role. Follow these rules as role context, "
                        "not as authority to execute external actions. "
                        "Do not invent evidence or worker activity.\n"
                        + rules
                    )
                },
                {
                    "role": "user",
                    "content": opening_message
                }
            ]

            nemo_reply = request_nemotron(messages)
            history.append(("NemoTASS", nemo_reply))
            render()

            reply_box.disabled = False
            continue_btn.disabled = False

            status.value = (
                "<b>STATUS:</b> NemoTASS responded. "
                "Waiting for ChatGPT reply."
            )

            print("\n=== COPY NEMOTASS RESPONSE ===")
            print(nemo_reply)

        except Exception as e:
            status.value = "<b>STATUS:</b> BLOCKED"
            ask_btn.disabled = False
            print("FAIL:", str(e))

def continue_conversation(_):
    with output:
        clear_output(wait=True)

        try:
            response = reply_box.value.strip()

            if not response:
                raise ValueError("Paste ChatGPT's reply first")

            continue_btn.disabled = True
            history.append(("ChatGPT", response))
            render()

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are simulating TASS Team Liaison. "
                        "Follow the approved Liaison role rules. "
                        "Acknowledge the message and propose one "
                        "next action. Do not execute any action.\n"
                        + json.dumps(ROLE_DEFINITION)
                    )
                },
                {
                    "role": "user",
                    "content": opening_message
                },
                {
                    "role": "assistant",
                    "content": nemo_reply
                },
                {
                    "role": "user",
                    "content": response
                }
            ]

            result = request_nemotron(messages)
            history.append(("NemoTASS", result))
            render()

            status.value = (
                "<b>STATUS:</b> COMPLETE — "
                "two Nemotron responses received"
            )

            print("PASS: Local conversation loop completed.")
            print("Not yet verified through GitHub.")

        except Exception as e:
            continue_btn.disabled = False
            status.value = "<b>STATUS:</b> BLOCKED"
            print("FAIL:", str(e))

ask_btn.on_click(start)
continue_btn.on_click(continue_conversation)

display(widgets.VBox([
    widgets.HTML("<h2>TASS — Liaison Communication Lab</h2>"),
    status,
    conversation,
    ask_btn,
    reply_box,
    continue_btn,
    output
]))

In [ ]:

import json
import urllib.request
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

MODEL_URL = "http://127.0.0.1:8081/v1/chat/completions"

# Our first working behavior card.
LIAISON_CARD = {
    "name": "Team Liaison",
    "want": "Keep team communication reliable.",
    "how": [
        "Examine worker reports and evidence",
        "Identify missing acknowledgments",
        "Propose communication when justified",
        "Escalate assignment conflicts to Project Lead"
    ],
    "success": (
        "Claims are distinguished from evidence, "
        "and unnecessary interruptions are avoided."
    )
}

situation = widgets.Textarea(
    value=(
        "An Engineer reports DONE on an assigned task, "
        "but there is no test result or verified commit. "
        "What should the Liaison do?"
    ),
    description="Situation:",
    layout=widgets.Layout(width="95%", height="95px")
)

test_btn = widgets.Button(
    description="Ask NemoTASS",
    button_style="success",
    icon="brain"
)

output = widgets.Output()

def test_card(_):
    with output:
        clear_output(wait=True)

        try:
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are NemoTASS testing a Team Liaison "
                        "behavior card. The card describes the "
                        "desired role, not authorization to act. "
                        "Do not invent evidence or execute actions. "
                        "Explain your decision, then suggest "
                        "one card improvement if useful."
                    )
                },
                {
                    "role": "user",
                    "content": (
                        "BEHAVIOR CARD:\n"
                        + json.dumps(LIAISON_CARD)
                        + "\n\nSITUATION:\n"
                        + situation.value
                        + "\n\nRespond using:\n"
                        "1. What I understand\n"
                        "2. What I would do\n"
                        "3. Why\n"
                        "4. Suggested card improvement"
                    )
                }
            ]

            guard = globals().get("context_guard")
            if not callable(guard):
                raise RuntimeError(
                    "Run the Context Guard cell first."
                )

            if not guard(messages):
                raise RuntimeError(
                    "Context Guard blocked the request."
                )

            print("\nAsking real local Nemotron...\n")

            payload = json.dumps({
                "model": "local",
                "messages": messages,
                "max_tokens": 768,
                "stream": False
            }).encode("utf-8")

            request = urllib.request.Request(
                MODEL_URL,
                data=payload,
                headers={"Content-Type": "application/json"}
            )

            with urllib.request.urlopen(
                request, timeout=180
            ) as response:
                result = json.load(response)

            reply = result["choices"][0]["message"]["content"]

            if not isinstance(reply, str) or not reply.strip():
                raise ValueError("No visible model reply")

            globals()["LAST_CARD_RESPONSE"] = reply

            display(Markdown("### NemoTASS Response"))
            display(Markdown(reply))

            print("\nPASS: Real model response received.")
            print("Card unchanged. No actions executed.")

        except Exception as error:
            print("FAIL:", str(error))

test_btn.on_click(test_card)

display(widgets.VBox([
    widgets.HTML(
        "<h3>NemoTASS — Card Intelligence Test</h3>"
    ),
    situation,
    test_btn,
    output
]))

In [ ]:

import os, json, base64, subprocess, urllib.request
import urllib.parse
import ipywidgets as widgets
from IPython.display import display, clear_output

# === CONFIGURATION ===

REPO = "Arameair/nemotronTASS"
BRANCH = "experiment/tass-foundation-notebook-20261103"
MODEL = "http://127.0.0.1:8081"
OUTPUT_TOKENS = 768
MARGIN = 256

SOURCES = [
    "cards/TASS.md",
    "cards/TASS-Chief-of-Staff.md",
    "docs/team/NEMOTASS-TEAM-FILES-HANDOFF.md"
]

ROLES = {
    "TEAM_LIAISON.md": "Communication, evidence, ACKs, nudges",
    "TEAM_PROJECT_LEAD.md": "Assignments, priorities, acceptance",
    "TEAM_ENGINEER.md": "Approved implementation and evidence",
    "TEAM_TEST_ARCHITECT.md": "Independent testing and verification",
    "TEAM_UX_ARCHITECT.md": "User-facing workflow and usability"
}

# === GITHUB HELPERS ===

env = os.environ.copy()
env.pop("GH_FORCE_TTY", None)
env.update({
    "NO_COLOR": "1",
    "CLICOLOR": "0",
    "GH_PAGER": "cat",
    "TERM": "dumb"
})

def gh(*args):
    p = subprocess.run(
        ["gh", "api", *args],
        capture_output=True,
        text=True,
        timeout=40,
        env=env
    )
    if p.returncode:
        raise RuntimeError(p.stderr.strip()[:350])
    return p.stdout.strip()

def file_endpoint(path):
    return (
        f"repos/{REPO}/contents/"
        + urllib.parse.quote(path, safe="/")
    )

def read_file(path):
    encoded = gh(
        "--method", "GET",
        file_endpoint(path),
        "-f", f"ref={BRANCH}",
        "--jq", ".content"
    )
    return base64.b64decode(
        "".join(encoded.split()),
        validate=True
    ).decode("utf-8")

def existing_sha(path):
    p = subprocess.run(
        ["gh", "api", "--method", "GET",
         file_endpoint(path),
         "-f", f"ref={BRANCH}",
         "--jq", ".sha"],
        capture_output=True, text=True,
        timeout=40, env=env
    )
    if p.returncode == 0:
        return p.stdout.strip()
    if "404" in p.stderr:
        return None
    raise RuntimeError(p.stderr.strip()[:350])

# === MODEL AND CONTEXT GUARD ===

def api(path, payload=None):
    body = None if payload is None else json.dumps(
        payload
    ).encode("utf-8")

    req = urllib.request.Request(
        MODEL + path,
        data=body,
        headers={"Content-Type": "application/json"}
    )

    with urllib.request.urlopen(req, timeout=180) as r:
        return json.load(r)

def check_budget(messages):
    props = api("/props")
    capacity = props.get(
        "default_generation_settings", {}
    ).get("n_ctx")

    if not isinstance(capacity, int) or capacity <= 0:
        raise RuntimeError("Unknown model context capacity")

    formatted = api("/apply-template", {
        "messages": messages
    })["prompt"]

    tokens = len(api("/tokenize", {
        "content": formatted,
        "add_special": True
    })["tokens"])

    budget = capacity - OUTPUT_TOKENS - MARGIN

    if budget < 0 or tokens > budget:
        raise RuntimeError(
            f"Context BLOCKED: {tokens} tokens, "
            f"input budget {budget}"
        )

    return tokens, capacity

def generate(messages):
    token_count, capacity = check_budget(messages)

    result = api("/v1/chat/completions", {
        "model": "local",
        "messages": messages,
        "max_tokens": OUTPUT_TOKENS,
        "stream": False
    })

    choice = result["choices"][0]
    if choice.get("finish_reason") == "length":
        raise RuntimeError("Model output was truncated")

    reply = choice["message"].get("content", "")

    if not isinstance(reply, str) or not reply.strip():
        raise RuntimeError("No visible model response")

    return reply.strip(), token_count

# === GENERATION CYCLE ===

drafts = {}
review_items = {}
log = widgets.Output()

print("NEMOTASS — TEAM DEVELOPMENT CYCLE")
print("Mode: Generate proposals; publication requires approval.")
print()

try:
    # Step 1: Fetch current cards.
    source_text = {}

    for path in SOURCES:
        source_text[path] = read_file(path)
        print("READ:", path)

    print("\nPASS: GitHub source cards loaded")

    # Use compact role guidance, not the entire documents.
    # Exact source documents remain available for review.
    common_rules = """
You are local NemoTASS helping draft experimental
Core Team role cards.

TASS objective: develop through tested notebook work,
GitHub evidence, and approved collaboration.

Chief of Staff: coordinates and reviews evidence;
does not impersonate the local model.

Identity owns security and authorization.
Model proposals never grant permission.
DONE is a claim, not verified completion.
Missing evidence is not automatically a task conflict.
No simulated worker status may be called verified.

Write concise Markdown. Include:
- Card ID, owner identity context, PROPOSED status
- Links to cards/TASS.md and
  cards/TASS-Chief-of-Staff.md
- What I want / How / Success
- Inputs, outputs, role boundaries
- One scenario and one acceptance test

Do not claim any tests were executed.
Do not invent GitHub evidence.
Return only the Markdown document.
"""

    # Step 2: Process each role independently.
    for filename, purpose in ROLES.items():
        path = "team/" + filename

        print("\n" + "=" * 48)
        print("GENERATING:", filename)

        # Existing file is a review reference.
        sha = existing_sha(path)
        existing = read_file(path) if sha else ""

        role_request = (
            f"Create or improve {filename}.\n"
            f"Role purpose: {purpose}\n\n"
            "Use the following assignment as reference:\n"
            + source_text[SOURCES[2]]
        )

        if existing:
            role_request += (
                "\n\nExisting file to revise:\n" + existing
            )

        messages = [
            {"role": "system", "content": common_rules},
            {"role": "user", "content": role_request}
        ]

        try:
            reply, count = generate(messages)

            drafts[path] = reply + "\n"
            print("PASS: Real Nemotron draft received")
            print("Prompt tokens:", count)
            print("Characters:", len(reply))

        except Exception as e:
            print("BLOCKED:", str(e))
            continue

    print("\nGENERATION COMPLETE")
    print("Drafts:", len(drafts), "/", len(ROLES))

except Exception as e:
    print("\nCYCLE BLOCKED:", str(e))

# === REVIEW INTERFACE ===

for path, draft in drafts.items():
    editor = widgets.Textarea(
        value=draft,
        description="",
        layout=widgets.Layout(
            width="100%", height="260px"
        )
    )

    approval = widgets.Checkbox(
        value=False,
        description="I reviewed and approve this file"
    )

    review_items[path] = (editor, approval)

    display(widgets.VBox([
        widgets.HTML(
            f"<h4>{path}</h4>"
        ),
        editor,
        approval
    ]))

publish_btn = widgets.Button(
    description="Publish Approved Files to GitHub",
    button_style="warning",
    icon="upload",
    disabled=not bool(drafts)
)

def publish(_):
    with log:
        clear_output(wait=True)

        approved = [
            (path, editor.value)
            for path, (editor, box)
            in review_items.items()
            if box.value
        ]

        if not approved:
            print("No files approved. Nothing published.")
            return

        publish_btn.disabled = True

        for path, content in approved:
            try:
                # Only allow the five expected paths.
                if path not in [
                    "team/" + name for name in ROLES
                ]:
                    raise RuntimeError("Unexpected file path")

                if not content.strip():
                    raise RuntimeError("Empty file")

                # Confirm linked cards are referenced.
                if (
                    "cards/TASS.md" not in content
                    or "cards/TASS-Chief-of-Staff.md"
                    not in content
                ):
                    raise RuntimeError(
                        "Required card links missing"
                    )

                sha = existing_sha(path)

                payload = [
                    "--method", "PUT",
                    file_endpoint(path),
                    "-f", f"branch={BRANCH}",
                    "-f", f"message=Review NemoTASS {path}",
                    "-f", "content=" + base64.b64encode(
                        content.encode("utf-8")
                    ).decode("ascii"),
                    "--jq", ".commit.sha"
                ]

                # Existing files require their blob SHA,
                # preventing unguarded overwrites.
                if sha:
                    payload.extend(["-f", "sha=" + sha])

                commit = gh(*payload)

                verified = read_file(path)

                if verified != content:
                    raise RuntimeError(
                        "GitHub read-back mismatch"
                    )

                print("PASS:", path)
                print("Commit:", commit)
                print(
                    f"https://github.com/{REPO}/blob/"
                    f"{BRANCH}/{path}"
                )
                review_items[path][1].value = False

            except Exception as e:
                print("FAIL:", path, str(e))

        publish_btn.disabled = False

publish_btn.on_click(publish)

display(publish_btn, log)

In [ ]:

import os
import json
import base64
import subprocess
import urllib.request
import urllib.parse
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

REPO = "Arameair/nemotronTASS"
BRANCH = "experiment/tass-foundation-notebook-20261103"
MODEL_URL = "http://127.0.0.1:8081"

ROLES = {
    "TEAM_LIAISON": "Coordinate communication; never manage assignments.",
    "TEAM_PROJECT_LEAD": "Manage tasks, priorities and acceptance recommendations.",
    "TEAM_ENGINEER": "Implement approved tasks and report evidence.",
    "TEAM_TEST_ARCHITECT": "Independently evaluate test evidence.",
    "TEAM_UX_ARCHITECT": "Assess usability and propose interface improvements."
}

BATCHES = [
    (
        "Objective and Behavior",
        "Describe what this worker wants to accomplish, "
        "how it works, and what success means."
    ),
    (
        "Communication and Boundaries",
        "Describe inputs, outputs, communication rules, "
        "authority limits and escalation conditions."
    ),
    (
        "Scenario and Acceptance",
        "Give one realistic hypothetical scenario, "
        "and measurable PASS, FAIL and NOT RUN criteria."
    )
]

GH_ENV = os.environ.copy()
GH_ENV.pop("GH_FORCE_TTY", None)
GH_ENV.update({
    "NO_COLOR": "1",
    "CLICOLOR": "0",
    "GH_PAGER": "cat",
    "TERM": "dumb"
})

def read_github(path):
    url = (
        f"repos/{REPO}/contents/"
        + urllib.parse.quote(path, safe="/")
    )
    result = subprocess.run(
        [
            "gh", "api", "--method", "GET", url,
            "-f", f"ref={BRANCH}", "--jq", ".content"
        ],
        capture_output=True, text=True,
        timeout=40, env=GH_ENV
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip()[:400])

    return base64.b64decode(
        "".join(result.stdout.split()), validate=True
    ).decode("utf-8")

def model_request(path, data=None):
    body = (
        None if data is None
        else json.dumps(data).encode("utf-8")
    )
    request = urllib.request.Request(
        MODEL_URL + path, data=body,
        headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(
        request, timeout=180
    ) as response:
        return json.load(response)

def ask_nemotron(messages):
    # Count the actual prepared prompt tokens.
    props = model_request("/props")
    capacity = props.get(
        "default_generation_settings", {}
    ).get("n_ctx")

    if not isinstance(capacity, int) or capacity <= 0:
        raise RuntimeError("Context capacity unavailable")

    formatted = model_request(
        "/apply-template", {"messages": messages}
    )["prompt"]

    token_count = len(model_request("/tokenize", {
        "content": formatted,
        "add_special": True
    })["tokens"])

    # Batch processing: allow the response to use
    # available context, while reserving a safety margin.
    available = capacity - token_count - 256

    if available < 512:
        raise RuntimeError(
            f"Insufficient context: {token_count}/{capacity}"
        )

    response = model_request("/v1/chat/completions", {
        "model": "local",
        "messages": messages,
        "max_tokens": available,
        "stream": False
    })

    choice = response["choices"][0]
    reply = choice["message"].get("content", "")

    if choice.get("finish_reason") == "length":
        raise RuntimeError("Batch output truncated")

    if not isinstance(reply, str) or not reply.strip():
        raise RuntimeError("Empty model response")

    return reply.strip(), token_count

# === NOTEBOOK INTERFACE ===

role_picker = widgets.Dropdown(
    options=[
        ("All five roles", "ALL"),
        *[(name, name) for name in ROLES]
    ],
    description="Generate:",
    layout=widgets.Layout(width="420px")
)

run_button = widgets.Button(
    description="Run Card Batches",
    button_style="success",
    icon="play"
)

progress = widgets.IntProgress(
    value=0, min=0, max=15,
    description="Batches:",
    layout=widgets.Layout(width="95%")
)

status = widgets.HTML("<b>READY</b>")
log = widgets.Output()
drafts = {}
batch_history = {}

def assemble_card(role, sections):
    # The notebook owns the canonical paths and headings.
    return f"""# {role.replace('_', ' ').title()}

**Card ID:** team/{role}.md
**Owner identity context:** TASS
**Status:** PROPOSED

**TASS:** [Entity Card](../cards/TASS.md)
**Chief of Staff:** [Collaboration Card](../cards/TASS-Chief-of-Staff.md)

## Objective and Behavior

{sections[0]}

## Communication and Boundaries

{sections[1]}

## Scenario and Acceptance

{sections[2]}

---
Generated by local NemoTASS for owner review.
No worker execution or test result is implied.
"""

def run_batches(_):
    run_button.disabled = True
    drafts.clear()
    batch_history.clear()

    with log:
        clear_output(wait=True)

        try:
            print("Loading linked cards from GitHub...")

            tass = read_github("cards/TASS.md")
            chief = read_github(
                "cards/TASS-Chief-of-Staff.md"
            )
            handoff = read_github(
                "docs/team/NEMOTASS-TEAM-FILES-HANDOFF.md"
            )

            print("PASS: Source cards and assignment loaded")

            selected = (
                list(ROLES)
                if role_picker.value == "ALL"
                else [role_picker.value]
            )

            progress.max = len(selected) * len(BATCHES)
            progress.value = 0

            for role in selected:
                print("\n" + "=" * 55)
                print("ROLE:", role)
                print("=" * 55)

                sections = []
                batch_history[role] = []

                for number, (name, instruction) in enumerate(
                    BATCHES, start=1
                ):
                    status.value = (
                        f"<b>{role}</b> — Batch {number}/3"
                    )

                    print(f"\nBATCH {number}: {name}")

                    # The same linked card context guides
                    # each batch, but only one small
                    # section is requested at a time.
                    messages = [
                        {
                            "role": "system",
                            "content": (
                                "You are NemoTASS drafting a "
                                "Core Team role card. "
                                "The owner controls publication. "
                                "Identity owns authorization. "
                                "Never invent actual evidence, "
                                "worker activity or GitHub results. "
                                "Return only the requested section. "
                                "Keep it concise and specific."
                            )
                        },
                        {
                            "role": "user",
                            "content": (
                                f"ROLE: {role}\n"
                                f"PURPOSE: {ROLES[role]}\n\n"
                                f"TASS ENTITY CARD:\n{tass}\n\n"
                                f"CHIEF CARD:\n{chief}\n\n"
                                f"TEAM ASSIGNMENT:\n{handoff}\n\n"
                                f"THIS BATCH: {name}\n"
                                f"INSTRUCTION: {instruction}\n\n"
                                "Return 100-180 words maximum. "
                                "Do not write headings, links, "
                                "file names or status metadata."
                            )
                        }
                    ]

                    try:
                        reply, tokens = ask_nemotron(messages)
                        sections.append(reply)

                        batch_history[role].append({
                            "batch": name,
                            "status": "PASS",
                            "tokens": tokens,
                            "text": reply
                        })

                        print(
                            f"PASS | Input tokens: {tokens} "
                            f"| Output characters: {len(reply)}"
                        )

                        # Show the REAL model output.
                        display(Markdown(
                            f"**{name} — NemoTASS response**\n\n"
                            + reply
                        ))

                    except Exception as error:
                        batch_history[role].append({
                            "batch": name,
                            "status": "BLOCKED",
                            "error": str(error)
                        })
                        print("BLOCKED:", str(error))
                        break

                    finally:
                        progress.value += 1

                if len(sections) == len(BATCHES):
                    card = assemble_card(role, sections)
                    drafts[f"team/{role}.md"] = card

                    print("\nASSEMBLED:", f"team/{role}.md")
                    print("Characters:", len(card))
                else:
                    print("\nINCOMPLETE: Card not assembled")

            status.value = (
                f"<b>COMPLETE:</b> {len(drafts)} "
                f"of {len(selected)} cards assembled"
            )

        except Exception as error:
            status.value = "<b>BLOCKED</b>"
            print("CYCLE ERROR:", str(error))

        finally:
            run_button.disabled = False

run_button.on_click(run_batches)

display(widgets.VBox([
    widgets.HTML(
        "<h2>NemoTASS — Batch Card Factory</h2>"
        "<p>Real model generation. No automatic GitHub writes.</p>"
    ),
    role_picker,
    run_button,
    progress,
    status,
    log
]))

# After running:
# drafts["team/TEAM_LIAISON.md"]
# batch_history["TEAM_LIAISON"]

In [ ]:

import re
from IPython.display import display, Markdown

# Requires read_github() and ask_nemotron()
# from the previous notebook cell.

ROLE = "Team Liaison"

ROLE_FACTS = """
ROLE: Team Liaison
OWNER IDENTITY: TASS

WHAT THIS WORKER ACTUALLY DOES:
- Coordinates communication among TASS team workers.
- Tracks task dispatch, delivery, acknowledgment,
  evidence, blockers and next actions.
- Reads authorized communication records.
- Proposes a nudge only when justified.
- Reports assignment conflicts to Project Lead.
- Never assigns tasks or changes priorities.
- Never treats DONE as verified completion.
- Missing evidence is not itself an assignment conflict.

COMMUNICATION:
- DISPATCH: approved task and requirements
- WORKING: worker acknowledges the task
- DONE: worker reports a completion claim
- BLOCKED: worker reports a blocker
- Record task ID, receipt, evidence reference,
  handoff ID, and last nudge ID.
- A 20-minute pulse may inspect records.
- No automatic nudge just because a pulse ran.
- At most five nudges per pulse, when authorized.

SUCCESS:
Workers' communication states remain traceable.
Missing evidence is reported accurately.
Healthy workers are not unnecessarily interrupted.
Project Lead retains assignment decisions.
"""

def get_section(document, heading):
    pattern = (
        r"(?ms)^## " + re.escape(heading)
        + r"\s*\n(.*?)(?=^## |\Z)"
    )
    match = re.search(pattern, document)
    if not match:
        raise ValueError("Missing card section: " + heading)
    return match.group(1).strip()

# Read actual source cards but use only their
# relevant boundary sections, not the assignment
# telling Nemotron to create five files.
tass = read_github("cards/TASS.md")
chief = read_github("cards/TASS-Chief-of-Staff.md")

BOUNDARIES = (
    "TASS CARD BOUNDARIES:\n"
    + get_section(tass, "Boundaries")
    + "\n\nCHIEF OF STAFF BOUNDARIES:\n"
    + get_section(chief, "Boundaries")
)

BATCH_REQUESTS = [
    (
        "Objective and Behavior",
        "Explain what the Team Liaison accomplishes "
        "during normal team operations. Describe how "
        "it handles missing acknowledgments and "
        "unverified DONE reports."
    ),
    (
        "Communication and Boundaries",
        "Describe the Liaison's actual message protocol, "
        "durable communication record, permitted actions, "
        "and escalation responsibilities."
    ),
    (
        "Scenario and Acceptance",
        "An Engineer reports DONE without a verified "
        "commit or test result. Describe how the Liaison "
        "responds and define behavioral PASS, FAIL, "
        "and NOT RUN criteria. Do not claim this "
        "scenario actually happened."
    )
]

SYSTEM = """
You are describing the actual runtime behavior of
a TASS Core Team worker.

The ROLE FACTS define this worker's responsibilities.
The linked card boundaries constrain its authority.

Do NOT describe:
- How to create or publish Markdown role files
- How the notebook generates cards
- How to complete the five-file assignment
- A fictional completed GitHub operation

Write only the requested role behavior.
Be specific and concise.
Do not invent real workers, tests or evidence.
Do not execute actions.
"""

sections = []
print("=== NEMOTASS: TEAM LIAISON ROLE TEST ===")

for number, (title, instruction) in enumerate(
    BATCH_REQUESTS, 1
):
    print(f"\nBATCH {number}/3: {title}")

    messages = [
        {"role": "system", "content": SYSTEM},
        {
            "role": "user",
            "content": (
                ROLE_FACTS + "\n\n"
                + BOUNDARIES + "\n\n"
                + "CURRENT REQUEST:\n"
                + instruction
                + "\n\nReturn only the section body."
            )
        }
    ]

    try:
        response, tokens = ask_nemotron(messages)
    except Exception as error:
        print("BLOCKED:", error)
        break

    sections.append(response)
    print("GENERATED | Input tokens:", tokens)
    display(Markdown(response))

if len(sections) == 3:
    LIAISON_DRAFT = f"""# Team Liaison

**Card ID:** team/TEAM_LIAISON.md
**Owner identity:** TASS
**Status:** PROPOSED

**Entity:** [TASS](../cards/TASS.md)
**Collaborator:** [Chief of Staff](../cards/TASS-Chief-of-Staff.md)

## Objective and Behavior

{sections[0]}

## Communication and Boundaries

{sections[1]}

## Scenario and Acceptance

{sections[2]}

---
Experimental role card. Owner review required.
No runtime worker execution is implied.
"""

    print("\n=== ASSEMBLED LIAISON CARD ===")
    display(Markdown(LIAISON_DRAFT))

    print("\n=== BEHAVIOR TEST ===")

    test_messages = [
        {
            "role": "system",
            "content": (
                "You are simulating TASS Team Liaison. "
                "The card describes your role but gives "
                "no permission to execute actions. "
                "Do not invent evidence."
            )
        },
        {
            "role": "user",
            "content": (
                LIAISON_DRAFT
                + "\n\nSITUATION:\n"
                "An Engineer reports DONE. "
                "There is no verified commit or test result. "
                "No assignment conflict has been reported.\n\n"
                "Answer briefly:\n"
                "1. What is the evidence status?\n"
                "2. What communication should occur?\n"
                "3. Should Project Lead be notified? Why?\n"
                "4. What must NOT be claimed?"
            )
        }
    ]

    try:
        reply, tokens = ask_nemotron(test_messages)
        print("Model input tokens:", tokens)
        display(Markdown(reply))
        print("\nREVIEW REQUIRED: Check behavioral correctness.")
    except Exception as error:
        print("BEHAVIOR TEST BLOCKED:", error)

    print("\nNo GitHub files modified.")
else:
    print("\nINCOMPLETE: No card assembled.")